# Milestone 1 — Data Preparation

Loads Monday (benign) and Wednesday (DoS/DDoS + Heartbleed + benign), deduplicates, and produces the shared train/val/test splits that both the autoencoder and RF-IDS will read from.

Run this notebook from its location in `notebooks/` — the next cell adds the project root to the path so `src` resolves correctly no matter where PyCharm sets the working directory.

In [9]:
import pandas as pd
import numpy as np

In [10]:
import sys
from pathlib import Path

# Make the project root importable regardless of PyCharm's working directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project


In [11]:
from src.data_prep import run_pilot_split

train, val, test = run_pilot_split(
    raw_dir=PROJECT_ROOT / "data" / "raw",
    out_dir=PROJECT_ROOT / "data" / "processed",
)

Loading Monday (benign)...
Loading Wednesday (DoS/DDoS + Heartbleed + benign)...
Deduplicating...
  dropped 26935 duplicate rows (529918 -> 502983)
  dropped 81909 duplicate rows (692703 -> 610794)
Splitting...

Final sizes -> train: 779641, val: 167063, test: 167073
  train label counts:
binary_label
BENIGN    644012
ATTACK    135629
  val label counts:
binary_label
BENIGN    138002
ATTACK     29061
  test label counts:
binary_label
BENIGN    138004
ATTACK     29069

Saved to /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/data/processed/{train,val,test}.csv


## Sanity check

Quick look at the splits before moving on to modeling.

In [12]:
print(train.shape, val.shape, test.shape)
train.head(50)

(779641, 81) (167063, 81) (167073, 81)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,day,binary_label
0,443,41420,3,1,43,6,31,6,14.333333,14.433757,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Monday,BENIGN
1,53,60462,1,1,42,106,42,42,42.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN
2,80,23172,1,1,6,6,6,6,6.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN
3,45166,85,1,1,0,0,0,0,0.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN
4,443,106115,2,0,12,0,6,6,6.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN
5,53,206,2,2,80,194,40,40,40.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN
6,443,117363386,16,15,1417,4544,778,0,88.562500,195.037422,...,54046.292610,303564,227131,5.830000e+07,3.684479e+05,58600000,58000000,BENIGN,Wednesday,BENIGN
7,63709,224,3,1,67,6,38,6,22.333333,16.010413,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Monday,BENIGN
8,56621,3,2,0,37,0,37,0,18.500000,26.162951,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Monday,BENIGN
9,53,60978,1,1,51,161,51,51,51.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN,Wednesday,BENIGN


In [13]:
from src.reliability_efficiency import run_all

results = run_all(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    models_dir=PROJECT_ROOT / "models",
)

  dropped 465 rows with inf/NaN feature values
  dropped 85 rows with inf/NaN feature values

--- Baseline (autoencoder), seed=42 ---
  epoch   1/30  train MSE: 0.231294
  epoch   5/30  train MSE: 0.052348
  epoch  10/30  train MSE: 0.039641
  epoch  15/30  train MSE: 0.022825
  epoch  20/30  train MSE: 0.026741
  epoch  25/30  train MSE: 0.021052
  epoch  30/30  train MSE: 0.019506

--- Baseline (autoencoder), seed=7 ---
  epoch   1/30  train MSE: 0.240177
  epoch   5/30  train MSE: 0.075141
  epoch  10/30  train MSE: 0.047038
  epoch  15/30  train MSE: 0.035217
  epoch  20/30  train MSE: 0.028452
  epoch  25/30  train MSE: 0.023596
  epoch  30/30  train MSE: 0.020161

--- Baseline (autoencoder), seed=123 ---
  epoch   1/30  train MSE: 0.237869
  epoch   5/30  train MSE: 0.063463
  epoch  10/30  train MSE: 0.030914
  epoch  15/30  train MSE: 0.054210
  epoch  20/30  train MSE: 0.030894
  epoch  25/30  train MSE: 0.022615
  epoch  30/30  train MSE: 0.026588

=== Baseline (autoencoder) 